# Weather-Station Behaviour Grouping for Automated Data Quality Assessment
**Academic ML project — K-Means clustering of NOAA weather stations (India)**

## 1. Introduction and problem statement
Weather stations differ in climate, altitude, instrumentation and reporting completeness. Given only their observations, can we group stations by *observed weather behaviour* in a reproducible, validated way, and are the groups stable and interpretable? The groups are **exploratory**; they are not permanent categories and make no claim about station reliability.

## Objectives
1. **Dataset preparation and preprocessing** of an open NOAA dataset.
2. **K-Means workflow**: several K, Euclidean vs Manhattan comparison, WCSS, silhouette, Davies-Bouldin, elbow, PCA.
3. **Cluster interpretation and validation**: profiles, stability, feature contributions, limitations.
4. **Reproducible end-to-end analysis**: fixed seed, relative paths, saved outputs.

**How to run:** `pip install -r requirements.txt`, then *Kernel → Restart & Run All*. The first code cell downloads data from NOAA (internet needed). Without internet, follow the manual instructions in `README.md` and set `SKIP_DOWNLOAD = True`.

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config, data_preprocessing as dp, feature_engineering as fe, clustering as cl
from src import evaluation as ev, visualization as viz, pipeline, report_generator as rg

warnings.filterwarnings("ignore", category=FutureWarning)
config.ensure_dirs()
SEED = config.SEED
np.random.seed(SEED)
pd.set_option("display.max_columns", 40, "display.width", 160)
SKIP_DOWNLOAD = False   # True = use files already present in data/raw/gsod_<year>/
print("Project root:", ROOT, "| seed:", SEED)

## 2. Dataset selection and source verification
**Dataset:** NOAA NCEI *Global Surface Summary of the Day (GSOD)*, derived from the Integrated Surface Hourly dataset.
- Landing page: https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=gov.noaa.ncdc:C00516
- Per-station CSV files: `https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/<YEAR>/<USAF><WBAN>.csv`
- Station metadata: https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv
- Documentation: https://www.ncei.noaa.gov/data/global-summary-of-the-day/doc/readme.txt

**Why suitable:** many stations; daily temperature, dew point, pressure, visibility, wind, precipitation and weather-event flags; public and unrestricted; supports station-level aggregation. **Selection rule:** stations with country code `IN` (India) whose ISD record spans the whole analysis year. No record is created or altered by hand; if a station file is unavailable it is simply absent.

**Units/structure:** one row = one station-day (observation-level). Temperatures in °F, wind in knots, precipitation in inches (converted to SI in cleaning). Missing values are coded with sentinels such as 9999.9 and 99.99.

## 3. Dataset loading

In [ ]:
raw = pipeline.step_load(config.YEAR, skip_download=SKIP_DOWNLOAD)
print("Raw shape:", raw.shape, "| stations:", raw["STATION"].nunique())
display(raw.head())

## 4. Exploratory data analysis (raw data)

In [ ]:
print("Date range:", raw["DATE"].min(), "->", raw["DATE"].max())
print("Duplicate station-days:", int(raw.duplicated(['STATION','DATE']).sum()))
print("Records per station (min / median / max):", raw.groupby('STATION').size().agg(['min','median','max']).tolist())
display(raw[["TEMP","DEWP","SLP","STP","VISIB","WDSP","MXSPD","MAX","MIN","PRCP"]].describe().T)

## 5. Data cleaning and preprocessing
Decisions (all counts are logged below): parse dates; drop exact duplicate station-days; convert GSOD sentinels to NaN; treat precipitation flag `I` as missing and flag `H` (0 reported but hourly precipitation seen = incomplete day, not a confirmed dry day) as ambiguous/missing, counted separately; convert to SI units; set physically implausible values to NaN; set inconsistent records (daily max < min, dew point > temperature) to NaN. **Plausible extreme values are kept** because genuine extremes carry weather information.

In [ ]:
clean, clean_log = pipeline.step_clean(raw)
display(clean_log)

## 6. Feature engineering and station-level aggregation
The clustering unit is the **station**. Daily series are aggregated into mean, standard deviation and extremes of temperature, humidity (derived from dew point with the Magnus formula), pressure, wind, visibility, observed precipitation total, rainy-day fraction/intensity (only for stations with >= 90 % valid precipitation coverage; otherwise unavailable, never scaled to 365 days) and *reported* fog/thunder frequencies (GSOD: 0 = no event or not reported, so these are lower bounds). Stations with fewer than 250 valid temperature days are excluded; the number, their characteristics and the resulting selection bias (clusters describe well-reporting stations only) are documented in the report, Section 7.1. Latitude, longitude, elevation and name are **not** clustering features (identifiers/context). Features missing for most stations or nearly duplicating another feature (|r| > 0.97) are dropped; no categorical variable remains after aggregation, so no encoding is needed.

In [ ]:
station_df, cols, sel_log, daily_missing, excluded, sens = pipeline.step_features(clean, config.YEAR)
print(len(station_df), "stations kept;", len(excluded), f"excluded (< {config.MIN_VALID_TEMP_DAYS} valid temperature days)")
display(sens); display(excluded.head(20))
display(sel_log)
display(station_df[["name","latitude","longitude","elevation_m"] + cols].head())

In [ ]:
station_missing = station_df[fe.WEATHER_FEATURES].isna().mean() * 100
viz.plot_missing(daily_missing, station_missing);
viz.plot_distributions(station_df, cols);
viz.plot_corr(station_df, [c for c in fe.WEATHER_FEATURES if c in station_df.columns]);

## 7. Imputation and standardisation
Remaining station-level gaps are **median-imputed** (robust to skew, keeps all stations). Features are **standardised** (z-score) because K-Means is distance-based and features have different units. Boxplots of the standardised features screen for outliers; no station is removed for being extreme.

In [ ]:
X, Xi = pipeline.step_matrix(station_df, cols)
print("Matrix:", X.shape, "| remaining NaN:", int(np.isnan(X).sum()))
viz.plot_boxplots(X, cols);

## 8. K-Means clustering across K and the distance-metric question
Standard K-Means (scikit-learn) minimises the **squared Euclidean** within-cluster sum of squares (WCSS) and does **not** support Manhattan distance. Two distinct Manhattan comparisons are therefore made:
1. **Evaluation only:** the silhouette of the *same* K-Means labels is computed with Manhattan distance. The algorithm is unchanged.
2. **Different algorithm:** **K-Medians** (assignment by Manhattan distance, centres = medians) implemented in `src/clustering.py`.

K-Medians has an L1 cost, not WCSS, so the two objectives are not comparable. Davies-Bouldin uses scikit-learn's Euclidean/mean-centroid definition for both.

In [ ]:
models, sweep = pipeline.step_sweep(X, SEED)
display(sweep[["method","k","objective","silhouette_euclidean","silhouette_manhattan","davies_bouldin","min_size","max_size"]])
viz.plot_silhouette(sweep);
viz.plot_db(sweep);
display(ev.kmeans_vs_kmedians(models))

## 9. Stability and selection of the final configuration
For every K: (a) **seed stability** — mean pairwise Adjusted Rand Index over 10 single-start runs with different seeds; (b) **subsample stability** — refit on 80 % of stations 30 times, assign all stations, ARI against the full-data clustering. The final K is chosen with a **composite** of silhouette, Davies-Bouldin, seed stability and subsample stability among K ≥ 3 whose smallest cluster has at least max(3, 5 % of stations); the elbow K is a tie-breaker. The highest silhouette alone is *not* used.

In [ ]:
stab = pipeline.step_stability(X)
K, elbow, scores = ev.select_final_k(sweep, stab, len(X))
display(scores)
print(f"Elbow K = {elbow} | Selected K = {K}")
viz.plot_elbow(sweep, elbow, K);

## 10. PCA visualisation of the selected configuration

In [ ]:
final = models[("kmeans", K)]
labels = cl.relabel_by_size(final.labels_)
centers = final.cluster_centers_[np.argsort(-np.bincount(final.labels_))]
fig, pca = viz.plot_pca(X, labels, K, centers, models[("kmedians", K)].labels_)
print("Variance explained by PC1, PC2:", (pca.explained_variance_ratio_[:2] * 100).round(1), "%")
viz.plot_sizes(labels);

## 11. Persist all results and generate the report
This cell re-runs the deterministic pipeline (same seed) and writes every table, figure and `final_report.md` using exactly the same functions as above. Sections 12–15 read the saved files.

In [ ]:
_backend = matplotlib.get_backend()
matplotlib.use("Agg", force=True)          # figures are saved; they are displayed below
summary = pipeline.run_all(config.YEAR, skip_download=True)
matplotlib.use(_backend, force=True)
print("Selected K:", summary["selected_k"], "| stations:", summary["n_stations"])
display(Markdown(rg.metric_agreement_text()))

## 12. Cluster profiling and interpretation

In [ ]:
sizes = pd.read_csv(config.PROFILE_DIR / "cluster_sizes.csv", index_col=0)
mean = pd.read_csv(config.PROFILE_DIR / "cluster_profile_mean.csv", index_col=0)
display(sizes); display(mean.T)
display(Image(config.FIG_DIR / "10_cluster_profile_heatmap.png"))
display(Image(config.FIG_DIR / "11_cluster_feature_comparison.png"))
display(Image(config.FIG_DIR / "13_station_map_by_cluster.png"))
display(Markdown(rg.cluster_narrative()))

*The text above is generated from the computed profiles. Read it as a description of how groups differ in observed annual statistics — not as a ranking of weather quality or station reliability.*

## 13. Feature contribution analysis
Eta-squared (share of a feature's variance explained by cluster membership) with ANOVA/Kruskal-Wallis statistics (descriptive only; p-values are optimistic because clusters were built from the same data), and **leave-one-feature-out** re-clustering (low ARI = clustering depends strongly on that feature).

In [ ]:
display(pd.read_csv(config.METRIC_DIR / "feature_contribution.csv"))
display(pd.read_csv(config.METRIC_DIR / "leave_one_feature_out.csv"))

## 14. Stability analysis
Stability across random initialisations, subsamples, neighbouring K and preprocessing/feature-set variants (robust scaler, KNN imputation, log-transformed precipitation, no pressure features, no extreme-value features, plus data-completeness features).

In [ ]:
display(pd.read_csv(config.METRIC_DIR / "stability_by_k.csv"))
display(pd.read_csv(config.METRIC_DIR / "cross_k_agreement.csv"))
display(pd.read_csv(config.METRIC_DIR / "preprocessing_variants.csv"))
display(Image(config.FIG_DIR / "12_stability.png"))

## 15. Limitations, bias and responsible use
- A single year of data; clusters may change in other years.
- NOAA's ISD/GSOD network is not a random sample of all Indian stations.
- Median imputation, standardisation and feature choice change distances (see variant analysis).
- K-Means favours compact, similar-sized, roughly spherical groups; climate regimes can be continuous.
- Pressure features mostly reflect elevation; derived humidity inherits measurement and rounding error.
- Clusters are **exploratory groupings**, not permanent station categories, and say nothing about instrument accuracy or reliability. Operational data-quality use needs expert review, station metadata and independent references.

## 16. Final verification checklist
| Objective | Where addressed |
|---|---|
| 1. Dataset preparation | Sections 2–7 (source, EDA, cleaning log, aggregation, selection, imputation, scaling) |
| 2. K-Means workflow | Sections 8–10 (K = 2–10, Euclid./Manhattan, WCSS, silhouette, DB, elbow, PCA, selection) |
| 3. Interpretation and validation | Sections 12–15 (profiles, contributions, stability, limitations) |
| 4. Reproducibility | Fixed seed, relative paths, `run_pipeline.py`, saved CSVs/figures, README |